In [ ]:
from voice_forge.omniconf import config
import pandas as pd
from pathlib import Path
import json
import re

ROOT = Path(config.yt_stt_output)

In [ ]:
UPLOADER_NAME = ''

In [3]:
def get_df(folder_name: str) -> pd.DataFrame:
    data = []
    for path in ROOT.joinpath(folder_name).rglob("*.json"):
        with open(path, "r") as f:
            data.append(json.load(f))

    return pd.DataFrame(data)


stt_data = []
for path in ROOT.joinpath("yt_captions").rglob("*.json3"):
    stt_data.append({"id": path.stem.split(".en")[0], "stt_path": path})

yt_stt_df = pd.DataFrame(stt_data)
yt_info_df = get_df("yt_info")

In [4]:
def stt_to_flat(json_data, remove_music=True, min_duration_ms=1000):
    """
    Converts YouTube STT-style JSON into a flat list of utterances.

    Args:
        json_data (dict): Parsed JSON input
        remove_music (bool): Remove segments containing [Music]
        min_duration_ms (int): Filter out segments shorter than this

    Returns:
        list of dicts with fields:
        - utterance
        - start_ms
        - end_ms
        - duration_ms
    """

    flat = []

    for evt in json_data.get("events", []):
        start = evt.get("tStartMs")
        duration = evt.get("dDurationMs")

        # skip invalid segments
        if start is None or duration is None:
            continue

        end = start + duration

        # concatenate text fragments inside segs
        text = " ".join(seg.get("utf8", "") for seg in evt.get("segs", []))
        text = text.strip()

        # normalize whitespace
        text = re.sub(r"\s+", " ", text)

        # skip empty
        if not text:
            continue

        # optionally remove [Music] segments
        if remove_music and re.fullmatch(r"\[Music\]", text, flags=re.IGNORECASE):
            continue

        # optionally filter very short duration segments
        if duration < min_duration_ms:
            continue

        flat.append(
            {
                "utterance": text,
                "start_ms": start,
                "end_ms": end,
                "duration_ms": duration,
            }
        )

    return flat


def prepare_hf_dataset(df, output_dir: Path = Path(config.yt_tts_hf_base)):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # 1. Write video metadata
    # -------------------------
    video_meta_path = output_dir / "metadata.jsonl"
    keep_cols = [
        "id",
        "title",
        "uploader",
        "categories",
        "age_limit",
        "tags",
        "comment_count",
        "chapters",
        "audio_channels",
        "duration",
    ]

    with video_meta_path.open("w") as f:
        for _, row in df.iterrows():
            meta = {col: row[col] for col in keep_cols}
            f.write(json.dumps(meta) + "\n")

    # -------------------------
    # 2. Write utterances per video
    # -------------------------
    utterances_root = output_dir / "utterances"
    utterances_root.mkdir(exist_ok=True)

    for _, row in df.iterrows():
        video_id = row["id"]
        flat = row["tts_flat__no_audio"]

        video_dir = utterances_root / video_id
        video_dir.mkdir(exist_ok=True)

        utt_path = video_dir / "utterances.jsonl"

        with utt_path.open("w") as f:
            for idx, seg in enumerate(flat):
                rec = {
                    "video_id": video_id,
                    "utt_id": f"{video_id}_{idx}",
                    "utterance": seg["utterance"],
                    "start_ms": seg["start_ms"],
                    "end_ms": seg["end_ms"],
                    "duration_ms": seg["duration_ms"],
                }
                f.write(json.dumps(rec) + "\n")

    # -------------------------
    # 3. Create processed.jsonl if missing
    # -------------------------
    processed_path = output_dir / "processed.jsonl"
    processed_path.touch(exist_ok=True)  # creates empty file if missing

    # -------------------------
    # 4. Create split_audio folder (empty)
    # -------------------------
    split_audio_dir = output_dir / "split_audio"
    split_audio_dir.mkdir(exist_ok=True)

    print(f"✅ HF dataset initialized at: {output_dir}")

In [ ]:
yt_combined = yt_stt_df.merge(yt_info_df, how="left")
yt_combined__affirmations = (
    yt_combined[yt_combined["uploader"] == UPLOADER_NAME]
    .dropna()
    .reset_index(drop=True)
)
yt_combined__affirmations["stt_path__char_len"] = yt_combined["stt_path"].apply(
    lambda x: len(x.read_text())
)
yt_combined__affirmations["tts_flat__no_audio"] = yt_combined__affirmations[
    "stt_path"
].apply(lambda x: stt_to_flat(json.loads(x.read_text())))
prepare_hf_dataset(yt_combined__affirmations)